# Semana 6 — Challenge 2: entrenamiento y selección del modelo

**Entradas:** `train.csv`, `data_contract.json`
**Salidas:** `champion_model.joblib`, `training_metadata.json`, `cv_results.csv`

> Reglas: todo el preprocesamiento va **dentro de los pipelines**, la selección se hace con **validación cruzada** y **no se carga `test.csv`**.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [ ]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

## 2. Pipeline de procesamiento

In [ ]:
feature_columns = contract["features"]

# TODO 1: preprocesadores  --> RESUELTO
# Para arboles: solo imputacion por mediana (no necesitan escalado)
tree_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), feature_columns),
])

# Para modelos sensibles a la escala: imputacion + StandardScaler
scaled_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns),
])

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si escaláramos **antes** de la validación cruzada, el `StandardScaler` calcularía la media y la desviación usando **todos** los datos, incluidos los que luego actúan como fold de validación. Eso es **data leakage**: información del conjunto de validación se filtra al entrenamiento y las métricas de CV salen optimistas y poco realistas. Al poner el scaler **dentro del pipeline**, en cada fold se ajusta solo con los datos de entrenamiento de ese fold y se aplica al de validación, reproduciendo fielmente lo que ocurrirá en producción con datos nuevos.

## 3. Modelos candidatos

In [ ]:
# TODO 2: pipelines para DecisionTree, LogisticRegression y KNN  --> RESUELTO
models = {
    "decision_tree": Pipeline([
        ("prep", tree_preprocessor),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("prep", scaled_preprocessor),
        ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("prep", scaled_preprocessor),
        ("clf", KNeighborsClassifier()),
    ]),
}
models

## 4. Comparación inicial con validación cruzada

In [ ]:
# TODO 3: StratifiedKFold(5) + cross_validate  --> RESUELTO
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

initial_comparison = {}
for name, pipe in models.items():
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring="accuracy", return_train_score=True,
    )
    initial_comparison[name] = {
        "train_acc_mean": scores["train_score"].mean(),
        "val_acc_mean": scores["test_score"].mean(),
        "val_acc_std": scores["test_score"].std(),
    }

initial_df = pd.DataFrame(initial_comparison).T.sort_values("val_acc_mean", ascending=False)
print("Comparación inicial (sin tuning):")
initial_df.round(4)

## 5. Optimización de hiperparámetros

In [ ]:
# TODO 4: GridSearchCV por modelo  --> RESUELTO
param_grids = {
    "decision_tree": {
        "clf__max_depth": [2, 3, 4, 5, None],
        "clf__min_samples_leaf": [1, 2, 4],
    },
    "logistic_regression": {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__penalty": ["l2"],
    },
    "knn": {
        "clf__n_neighbors": [3, 5, 7, 9, 11],
        "clf__weights": ["uniform", "distance"],
    },
}

searches = {}
tuned_comparison = {}
for name, pipe in models.items():
    gs = GridSearchCV(pipe, param_grids[name], scoring="accuracy", cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train)
    searches[name] = gs
    tuned_comparison[name] = {
        "best_cv_acc": gs.best_score_,
        "best_params": gs.best_params_,
    }
    print(f"{name:>22}: best CV acc = {gs.best_score_:.4f} | {gs.best_params_}")

## 6. Selección y persistencia del campeón

In [ ]:
# TODO 5: campeón = mayor best_score_ de CV, persistir artefactos  --> RESUELTO
champion_name = max(tuned_comparison, key=lambda n: tuned_comparison[n]["best_cv_acc"])
champion_search = searches[champion_name]
champion = champion_search.best_estimator_

print("Campeón:", champion_name)
print("CV accuracy:", round(champion_search.best_score_, 4))

# Guardar el pipeline COMPLETO (preprocesamiento + modelo)
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "cv_accuracy": float(champion_search.best_score_),
    "best_params": champion_search.best_params_,
    "random_state": RANDOM_STATE,
    "models_compared": list(models.keys()),
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2), encoding="utf-8"
)

# cv_results.csv con el resumen de las tres familias
cv_rows = []
for name, gs in searches.items():
    cv_rows.append({
        "model": name,
        "best_cv_accuracy": gs.best_score_,
        "best_params": json.dumps(gs.best_params_),
    })
pd.DataFrame(cv_rows).sort_values("best_cv_accuracy", ascending=False)\
    .to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print("\nArtefactos guardados.")

In [ ]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

**Justificación de la selección:**

> Se elige el modelo con mayor **accuracy de validación cruzada** (sin mirar test). En el dataset Wine, tanto la **regresión logística** como el **KNN** (ambos con `StandardScaler`) suelen alcanzar accuracy de CV muy alto (≈0.97–0.99), por encima del árbol de decisión, porque las clases son casi linealmente separables tras estandarizar. La decisión final considera: **desempeño** (accuracy de CV), **variabilidad** (desviación estándar entre folds — se prefiere un modelo estable), **sobreajuste** (la brecha entre train y validation; el árbol sin profundidad limitada tiende a sobreajustar), **complejidad** y **interpretabilidad**. La regresión logística es un buen campeón porque combina alto desempeño, baja varianza y coeficientes interpretables.

## Checklist
- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.